# 01 — AequilibraE projects and networks

**AequilibraE** is a fully-featured, open-source transportation modeling package for Python.
This notebook series walks through a complete transport-modeling workflow, using
**lonboard** for interactive, fully offline WebGL maps inside JupyterLab.

In this first notebook we:

1. create a model from a bundled example (Coquimbo/La Serena, Chile);
2. look at how an AequilibraE *project* is structured (a SQLite/SpatiaLite database);
3. load the network's links, nodes and zones as GeoDataFrames;
4. put everything on an interactive map.

> **Note on GIS files** — an AequilibraE project is a standard *SpatiaLite* database.
> This fork reads and writes that format with a pure-Python engine
> (shapely + pyproj + SQLite's built-in R\*Tree index), so installing the fork's wheel
> is all you need — no `mod_spatialite` system package, on any OS. The files remain
> 100% compatible with QGIS and any other SpatiaLite-aware tool.

**Requirements**: this fork of AequilibraE plus `jupytergis` — see the [setup instructions](README.md). Do **not** `pip install aequilibrae` from PyPI: that installs the upstream package, which still needs native SpatiaLite. Run inside JupyterLab to see the maps.

In [1]:
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4

from aequilibrae.utils.create_example import create_example

# Every notebook in this series works inside a throw-away folder
fldr = str(Path(gettempdir()) / uuid4().hex)

# 'coquimbo' is a real-world model of Coquimbo/La Serena, Chile.
# Other options: 'sioux_falls' (the classic toy network) and 'nauru'.
project = create_example(fldr, "coquimbo")
project

## What is inside a project?

The project folder holds a handful of files — the two SQLite databases are the model itself:

- `project_database.sqlite` — network (links, nodes), zones, matrix index, results index…
- `public_transport.sqlite` — GTFS-derived transit network (created on demand);
- `matrices/` — demand and skim matrices (OMX / AEM files);
- `parameters.yml` — model parameters.

Because it is *just SQLite*, you can open it with any SQL client — or QGIS.


In [2]:
import pandas as pd

with project.db_connection as conn:
    tables = pd.read_sql("SELECT name, type FROM sqlite_master WHERE type IN ('table','view') ORDER BY name", conn)
tables[~tables.name.str.startswith(("idx_", "sqlite_", "geometry_", "spatial_", "views_", "virts_"))].head(20)

,name,type
0,ElementaryGeometries,table
1,KNN,table
2,SpatialIndex,table
3,about,table
4,attributes_documentation,table
5,data_licenses,table
6,geom_cols_ref_sys,view
24,link_types,table
25,links,table
26,matrices,table


## The network

`project.network` gives access to links and nodes. The `.data` accessors return
**GeoDataFrames** (WGS84 / EPSG:4326), which makes the whole geopandas/shapely
ecosystem available directly.


In [3]:
links = project.network.links.data
nodes = project.network.nodes.data
zones = project.zoning.data

print(f"{len(links)} links, {len(nodes)} nodes, {len(zones)} zones")
links[["link_id", "a_node", "b_node", "direction", "distance", "modes", "link_type"]].head()

19983 links, 15724 nodes, 133 zones


,link_id,a_node,b_node,direction,distance,modes,link_type
0,1,64158,64194,0,15.192014,ct,residential
1,2,64208,64194,1,117.170146,ct,residential
2,3,47501,47539,1,37.343065,ct,residential
3,12,78052,78051,1,47.825940,ct,residential
4,13,73608,79808,1,93.536834,ct,residential


In [4]:
# Modes available in this model, straight from SQL
with project.db_connection as conn:
    modes = pd.read_sql("SELECT mode_name, mode_id, description FROM modes", conn)
modes

,mode_name,mode_id,description
0,car,c,All motorized vehicles
1,transit,t,Public transport vehicles
2,walk,w,Walking links
3,bicycle,b,Biking links


## Interactive map with JupyterGIS

`GISDocument` builds a collaborative map document rendered by JupyterLab.
Every layer you add appears in the layer tree on the left of the map widget,
where symbology can also be edited interactively.


In [5]:
# Maps, cartographic standards and UK geography helpers.
# Model logic stays in the notebook; everything reusable lives in notebooks/uktools/.
from uktools import *


In [6]:
# field()/constant() symbology builders come from the map helper cell

doc = new_map(links, zoom=12)

add_gdf(doc, zones, "zones", opacity=0.4, symbology=[[constant("#f59e0b").encoding("fill")]])
add_gdf(doc, links, "links", symbology=[[constant("#1d4ed8").encoding("stroke")]])
add_gdf(doc, nodes[nodes.is_centroid == 1], "centroids", symbology=[[constant("#dc2626").encoding("fill")]])

doc  # render the map (interactive in JupyterLab)

[interactive offline map - run the notebook to display]

Try the layer tree: toggle layers, right-click one to open the *Symbology* editor,
or use `doc.export_to_qgis("model.qgz")` to hand the exact same map to QGIS.

## Closing up

Always close a project when done — it releases the SQLite connections and flushes logs.


In [7]:
project.close()

---
**Next:** [02 — Zones and centroid connectors](02_zones_and_connectors.ipynb), where we
build a zoning system from scratch and hook it to the network.
